# Module 8.4 — Multi-Agent RAG Patterns

Complex knowledge tasks often require **multiple specialised agents** working together:

```
                  ┌─────────────────────┐
    User Query ──▶│  Orchestrator Agent │
                  └──────┬──────┬───────┘
                         │      │
               ┌─────────▼─┐ ┌──▼──────────┐
               │ Retrieval │ │  Web Search │
               │  Agent    │ │   Agent     │
               └─────────┬─┘ └──┬──────────┘
                         │      │
                  ┌──────▼──────▼──────┐
                  │  Synthesis Agent   │
                  └────────────────────┘
```

## Patterns Covered
1. **Router + Specialist** — route query to the right knowledge base
2. **Sequential agents** — researcher → writer → validator
3. **Parallel retrieval** — multiple sources simultaneously

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
from langchain.tools.retriever import create_retriever_tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm        = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# ── Two specialised knowledge bases ──────────────────────────────────────────
tech_docs = [
    Document(page_content='Python 3.12 introduces the new type parameter syntax for generics.'),
    Document(page_content='LangChain supports async execution via ainvoke and astream.'),
    Document(page_content='Docker containers package applications with all their dependencies.'),
]
tech_vs   = Chroma.from_documents(tech_docs, embeddings, collection_name='tech_kb')
tech_tool = create_retriever_tool(
    tech_vs.as_retriever(search_kwargs={'k': 2}),
    name='tech_search',
    description='Search technical documentation about Python, LangChain, Docker.'
)

biz_docs  = [
    Document(page_content='Q3 revenue grew 23% year-over-year driven by cloud segment.'),
    Document(page_content='Customer acquisition cost decreased by 12% following new onboarding flow.'),
    Document(page_content='Churn rate for enterprise tier dropped to 2.1% in Q3.'),
]
biz_vs    = Chroma.from_documents(biz_docs, embeddings, collection_name='biz_kb')
biz_tool  = create_retriever_tool(
    biz_vs.as_retriever(search_kwargs={'k': 2}),
    name='business_search',
    description='Search business metrics, revenue, customer data.'
)

print('✅ Specialised knowledge bases ready')

In [ ]:
# ── Pattern 1: Router Agent ───────────────────────────────────────────────────
router_prompt = ChatPromptTemplate.from_template("""
You are a routing agent. Decide which tool to use.
Available tools:
- tech_search: for Python, Docker, LangChain questions
- business_search: for revenue, customers, business metrics
- none: for general questions not needing retrieval

Answer only with the tool name or 'none'.
Question: {question}
""")

def router_agent(question: str) -> str:
    tool_name = (router_prompt | llm | StrOutputParser()).invoke({'question': question}).strip().lower()
    print(f'  🔀 Router chose: {tool_name}')

    if tool_name == 'tech_search':
        docs = tech_vs.similarity_search(question, k=2)
    elif tool_name == 'business_search':
        docs = biz_vs.similarity_search(question, k=2)
    else:
        docs = []

    context = '\n'.join(d.page_content for d in docs) if docs else 'No retrieval needed.'
    gen_prompt = ChatPromptTemplate.from_template(
        'Answer using context (if available).\nContext: {context}\nQuestion: {question}'
    )
    return (gen_prompt | llm | StrOutputParser()).invoke({'context': context, 'question': question})

for q in [
    'What is new in Python 3.12?',
    'How did revenue perform in Q3?',
    'What is the capital of Australia?',
]:
    print(f'\nQ: {q}')
    ans = router_agent(q)
    print(f'A: {ans[:150].strip()}')

In [ ]:
# ── Pattern 2: Sequential Researcher → Writer → Validator ─────────────────────
import concurrent.futures

def researcher(question: str) -> str:
    docs    = tech_vs.similarity_search(question, k=3)
    context = '\n'.join(d.page_content for d in docs)
    prompt  = ChatPromptTemplate.from_template(
        'Summarise the key facts from this context relevant to the question.\nContext: {context}\nQuestion: {question}'
    )
    return (prompt | llm | StrOutputParser()).invoke({'context': context, 'question': question})

def writer(research: str, question: str) -> str:
    prompt = ChatPromptTemplate.from_template(
        'Write a clear, concise answer using these research notes.\nNotes: {research}\nQuestion: {question}'
    )
    return (prompt | llm | StrOutputParser()).invoke({'research': research, 'question': question})

def validator(answer: str, question: str) -> dict:
    prompt = ChatPromptTemplate.from_template(
        'Is this answer accurate and complete? Score 1-5 and explain briefly.\nQ: {question}\nA: {answer}\nScore and reason:'
    )
    feedback = (prompt | llm | StrOutputParser()).invoke({'question': question, 'answer': answer})
    return {'answer': answer, 'feedback': feedback}

# ── Run sequential pipeline ───────────────────────────────────────────────────
q = 'How does LangChain support async operations?'
print(f'Question: {q}\n')

research = researcher(q)
print(f'📚 Researcher: {research[:120].strip()}...')

draft = writer(research, q)
print(f'✍  Writer   : {draft[:120].strip()}...')

validation = validator(draft, q)
print(f'🔍 Validator: {validation["feedback"][:120].strip()}...')

In [ ]:
# ── Pattern 3: Parallel retrieval from multiple KBs ──────────────────────────
def parallel_retrieve(question: str) -> list[Document]:
    """Retrieve from tech and business KBs in parallel, merge results."""
    with concurrent.futures.ThreadPoolExecutor() as executor:
        tech_future = executor.submit(tech_vs.similarity_search, question, 2)
        biz_future  = executor.submit(biz_vs.similarity_search,  question, 2)
        tech_docs   = tech_future.result()
        biz_docs    = biz_future.result()
    return tech_docs + biz_docs

q    = 'How can technology help improve business metrics?'
docs = parallel_retrieve(q)
print(f'Parallel retrieval for: "{q}"')
print(f'Total docs retrieved: {len(docs)}')
for d in docs:
    print(f'  • {d.page_content[:80].strip()}')